In [1]:
import os
from dash import Dash, html, dcc, callback, Output, Input, no_update, State
import dash_ag_grid as dag
import pandas as pd
import plotly.express as px
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash_auth
from dash_auth import BasicAuth
import math
from scipy import stats
import numpy as np
from sqlalchemy import create_engine, text

In [192]:
VALID_USERNAME_PASSWORD_PAIRS = {
    'dbuser': 'pa$$w0rd'
}

In [229]:
# drug name, cell line and mysql table dictionary
DATASETS = {
    'INX315': {
        'MB157': 'MB157_WT',
        'MCF7': 'MCF7',
        '1222': '1222',
        '3226': '3226',
        'MiaPaCa_2': 'MiaPaCa_2',
        'KURAMOCHI': 'KURAMOCHI'
    },
    'Abema': {
        'MCF7_RB_del': 'MCF7_RB_del_Abema',
        'MCF7': 'MCF7_Abema',
        'T47D': 'T47D_Abema',
        '1222': '1222_Abemaciclib',
        '226': '226_Abema'
    },
    'Palbo': {
        '226': '226_Palbo',
        'HCC1806': 'HCC1806_Palbo',
        'T47D': 'T47D_Palbo',
        'MCF7': 'MCF7_Palbo',
        '3226': '3226_Palbo'        
    }
}

In [230]:
#  DATABASE CONNECTION SETUP
USER = "vectra"
PASSWORD = "ritho9Ng"
HOST = "10.126.0.41"
PORT = "3306"
DATABASE = "drug_screen"

# Create a reusable SQLAlchemy connection engine
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

In [231]:

# Initialize the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

In [232]:

left_pane = [
    dbc.CardBody(
        [
            
            html.P(
                "1). Select a Drug/Condition",
            ),
            dcc.Dropdown(
                id='drug-name-dropdown',
                options=[{'label': drug, 'value': drug} for drug in DATASETS.keys()],
                placeholder="Select a drug/treatment...",
                clearable=True,
            ),
            html.Br(),
            html.P(
                "2). Select Cell Line(s)",
            ),
            dcc.Dropdown(
                id='cell-line-dropdown',
                placeholder="Select cell line(s)...",
                clearable=True,
                multi=True,
            ),
            html.Br(),     
            dcc.Markdown(
                """
                Navigate by clicking the tabs on the right side panel. Currently, only the 
                aggregated pheno type data and marker intensity data at single cell level 
                are availble. This resource is still under active development...
                """
            ),
       
        ],
    ),
]

card_growth_rate = dbc.Card([
    dbc.CardHeader(html.H4("Cell Growth Line Plots")),
    dbc.CardBody([
        dbc.Row([
            dbc.Col([
                dcc.Graph(id='cell-growth-lineplot'),], width=12),
        ]),
        
    ]),
    dbc.CardFooter([dcc.Markdown(
                """
                Vertical bars on each line are standard errors of mean (SEM).
                """
            ),]),
    

], className="shadow")



footer_markdown=dcc.Markdown("""
    This is the footer text...
""")

In [233]:
app.layout = dbc.Container([
    dbc.Row(html.H1("Explore Drug Screen Data Sets"), style = {'textAlign' : 'center'}),
    
     dbc.Row([
        dbc.Col(dbc.Row(dbc.Card(left_pane, color="light")), width=2),
        dbc.Col([
            dbc.Tabs([
                dbc.Tab(label='Treatment Effects', tab_id="growth_tab", children=[  
                    card_growth_rate, 
                ]),
                dbc.Tab(label='Download Data', tab_id="single_tab", children=[
                    #card_growth_table,
                    
                ]),               
            ],
            id="card-tabs",
            active_tab="growth_tab", 
            ),           
        ], width=10),
    ]),
    dbc.Row(footer_markdown),
], fluid=True)                          

In [234]:

# Populate cell line dropdowns based on selected drug/treatment
@app.callback(
    Output('cell-line-dropdown', 'options'),
    Input('drug-name-dropdown', 'value')
)
def cell_line_dropdowns(drug_name):
    if not drug_name:
        return []
    
    cell_dict = DATASETS[drug_name]
    options = [{'label': cell, 'value': cell} for cell in cell_dict.keys()]
       
    return options

@app.callback(
    Output('cell-growth-lineplot', 'figure'),
    State('drug-name-dropdown', 'value'),
    Input('cell-line-dropdown', 'value'),
    prevent_initial_call=True,
)
def updata_lineplot(selected_condition, selected_cells):
    if not selected_condition or not selected_cells:
        fig = go.Figure()
        fig.update_layout(title="No data selected. Please select a drug and cell line(s).", template="plotly_white")
        return fig
        
    # Initialize a list to pool data frames fetched from individual tables
    collected_data = []
    
    # Establish a clean context connection to the database
    with engine.connect() as conn:
        for cell in selected_cells:
            
            # 1. get table name
            table_name = DATASETS[selected_condition][cell]
                
            try:
                # 2. Build explicit query utilizing the text abstraction to secure execution
                # Wrapped columns with backticks to prevent reserved word conflicts like `Condition`
                query = text(f"SELECT * FROM `{table_name}`")
                    
                # 3. Stream data from SQL table straight into a pandas DataFrame
                df_table = pd.read_sql(query, conn)
                    
                if not df_table.empty:
                    # Inject label tracking metadata back into the frame
                    df_table["id"] = selected_condition + " - " + cell
                    #df_table["cell"] = cell
                    collected_data.append(df_table)
                        
            except Exception as e:
                # Silent warning/bypass if a specific table combo doesn't exist in your database yet
                print(f"Table {table_name} could not be loaded or does not exist. Error: {e}")
                continue

    # If no data found across combinations, return empty chart alert
    if not collected_data:
        fig = go.Figure()
        fig.update_layout(title="No matching tables found in the database.", template="plotly_white")
        return fig

    # Merge all separate SQL table data frames together into a single master sheet
    df_master = pd.concat(collected_data, ignore_index=True)

    # 4. Group data and calculate Mean and Standard Error (SEM) across replicates
    df_summary = (
        df_master.groupby(["id", "Time", "Condition"])["fold_change"]
        .agg(Mean="mean", StdErr=lambda x: stats.sem(x, ddof=1) if len(x) > 1 else 0)
        .reset_index()
    )
    df_summary["StdErr"] = df_summary["StdErr"].fillna(0)

    # Configuration
    cols = 3  # Set your fixed number of columns
    total_plots = len(selected_cells)
    rows = math.ceil(total_plots / cols)

    # Initialize subplots
    fig = make_subplots(
        rows=rows, 
        cols=cols, 
        subplot_titles=list(selected_cells),
        vertical_spacing=0.2,  # Add space between rows
        horizontal_spacing=0.05
    )
    
    # Pull a high-contrast qualitative color palette array
    palette = px.colors.qualitative.Plotly

    # Define a custom hex-to-rgba converter for trace shading
    def hex_to_rgba(hex_str, opacity):
        hex_str = hex_str.lstrip('#')
        rgb = tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))
        
        return f"rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, {opacity})"


    # Loop with Row/Col Calculation
    #df_summary = df_summary.sort_values(by="cell")
    
    for idx, id in enumerate(df_summary.id.unique()):
        # Calculate current row and column (1-based)
        curr_row = (idx // cols) + 1
        curr_col = (idx % cols) + 1
        
        # Unique identifier names for each independent legend box
        legend_identifier = f"legend_{id}"
        
        if idx == 0:
            legend_identifier = 'legend'
        else:
            legend_identifier = f'legend{idx + 1}'
        
        for j, cond in enumerate(df_summary['Condition'].unique()):
            
            group_data = df_summary[(df_summary['id'] == id) & (df_summary['Condition'] == cond)].sort_values('Time')

            # Assign specific matching palette colors to this category group
            base_color_hex = palette[j % len(palette)]
            fill_color_rgba = hex_to_rgba(base_color_hex, opacity=0.15)
            
            fig.add_trace(go.Scatter(
                x=group_data['Time'],
                y=group_data['Mean'],
                mode='lines+markers',
                name=cond,
                showlegend=True,
                line=dict(color=base_color_hex, width=2.5),
                marker=dict(size=6, symbol='circle'),
                
                # --- CRITICAL LEGEND ROUTING CONFIGURATION ---
                legend=legend_identifier,       # Binds this trace to a custom legend box
                legendgroup=id,          # Groups related trace properties together
                error_y=dict(
                    type='data',                  # Tells Plotly to read values from an array
                    array=group_data['StdErr'],   # Sets the error bar heights above/below the mean
                    visible=True,                 # Renders the error bars visible
                    thickness=1.5,                # Line width of the error bars
                    width=4,                      # Width of the horizontal crossbar cap (T-bar)
                    color=base_color_hex               # Automatically matches the line color
                )
            ), row=curr_row, col=curr_col)
        
        
        # --- CRITICAL FIX: Dynamically read layout grid boundary coordinates ---
        # Finds the specific structural identifier labels used behind the scenes
        xaxis_key = 'xaxis' if idx == 0 else f'xaxis{idx + 1}'
        yaxis_key = 'yaxis' if idx == 0 else f'yaxis{idx + 1}'

        # Render the chart once to let Plotly automatically generate domain bounds [INDEX]
        # We pull the left domain line for X, and the top domain ceiling line for Y [INDEX]
        x_left_edge = fig.layout[xaxis_key].domain[0]
        y_top_ceiling = fig.layout[yaxis_key].domain[1]

        # Apply precise tracking positions directly to this specific legend object
        fig.update_layout({
            legend_identifier: dict(
                x=x_left_edge + 0.025,          # Aligns inside the left axis, adding 1.5% padding
                y=y_top_ceiling - 0.015,         # Aligns below the top axis, removing 1.5% padding
                xanchor="left",                  # Anchors left edge of box
                yanchor="top",                   # Suspends from the top edge
                bgcolor="rgba(255,255,255,0.75)",# Translucent background
                bordercolor="rgba(0,0,0,0.1)",
                borderwidth=1
            )
        })
        fig.layout.annotations[idx].text = id

    fig.update_layout(
        height=400 * rows, 
        width=400* cols,       
        template="plotly_white",
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="Time Points")
    fig.update_yaxes(title_text="Fold Change") #, row=1, col=1)
    
    return fig
    


if __name__ == '__main__':
    app.run(debug=True, host="10.126.0.41", port=8787)
